# **Deepfake Media Detection on FaceForensics++**

**Course:** CSCI 4701 - Deep Learning, Spring 2026

**Milestone:** 2

**Team:** Bilal Hasanov, Muaataz Abdulhakeem Ismaeel

## Project Goal

The goal of this project is to investigate whether deep CNNs can reliably distinguish between real faces and AI-manipulated face videos, using the [FaceForensics++ extracted frames](https://www.kaggle.com/datasets/adham7elmy/faceforencispp-extracted-frames) dataset. Due to the dataset being video-based yet most pretrained image classifiers operating on individual frames, we structure the project in three phases:

1. **Architecture comparison.** We fine-tune three backbones that were pretrained on the ImageNet dataset (ResNet-50, EfficientNet-B0, and EfficientNet-V2-S) on the FF++ frame-level binary problem (real vs fake) and pick the best performing model.
2. **Frame-level inference and video-level aggregation.** Take the winning model and use it to classify every frame in a test video, then aggregate those frame predictions to a single video-level decision/prediction using either majority voting or average softmax.
3. **Backbone as a feature extractor and classical ML.** First we drop the trained classification head, then embed every frame to a 1280 dimensional vector with the frozen backbone, use mean-pooling for the frame embeddings within each video, and finally train traditional ML classifiers (SVM, MLP, Logistic Regression) on those video-level features.

## Note about reproducibility

Fine-tuning each of the three backbones takes hours on a Kaggle T4. We are therefore not retraining inside this notebook. This notebook loads the three saved checkpoints from Drive and runs evaluation on the test set. The original training notebooks with preserved outputs are located in `notebooks/architecture_training/` and can be inspected directly. The `src/training.py` module captures the same training loop in a reusable form for anyone who wants to reproduce training from scratch.


## 1. Environment Setup

This notebook is designed to run end-to-end on **Google Colab** with a GPU runtime (T4 is sufficient). The setup below:

1. mounts Google Drive (where `src/` and the rest of this repo live),
2. installs `kagglehub` (used to pull the prepared FF++ split and the three trained checkpoints from Kaggle), and
3. configures Kaggle credentials so `kagglehub` can authenticate.

If you are running locally, replace the Drive mount with a `cd` into your local clone of the repo, install the same dependencies, and put your `kaggle.json` at `~/.kaggle/kaggle.json`.

In [2]:
# mount drive and cd into the repo folder so `from src import ...` works
from google.colab import drive
drive.mount('/content/drive')

# change this if you uploaded the repo to a different folder
%cd /content/drive/MyDrive/csci-4701-deepfake-detection

Mounted at /content/drive
/content/drive/MyDrive/csci-4701-deepfake-detection


In [3]:
# install kagglehub
!pip -q install --upgrade kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 9.9 MB/s eta 0:00:00


In [4]:
# (this is a throwaway kaggle account)
import os, json

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
creds = {
    "username": "delno002",
    "key": "KGAT_796610068c0bfe9926a88925cdbc9334"
}
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(creds, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("kaggle credentials configured")

kaggle credentials configured


## 2. Configuration

This cell pulls the prepared FF++ split from Kaggle via `kagglehub` (it gets cached locally on first download, so re-running the cell within the same Colab session is instant) and points the project config at the three trained checkpoint files in your Drive folder.

The actual fine-tuning of the three architectures takes hours per architecture and was done in separate Kaggle notebooks. Those notebooks are preserved under `notebooks/architecture_training/` and the saved weights are read by this notebook from Drive.

In [9]:
from pathlib import Path
import kagglehub
from src import config

split_path = kagglehub.dataset_download("delno002/split-ffpp")
print("split downloaded to:", split_path)

DOWNLOADABLES = Path("/content/drive/MyDrive/Downloadables")

config.DATA_DIR = Path(split_path)
config.TRAIN_DIR = config.DATA_DIR / "train"
config.VAL_DIR = config.DATA_DIR / "val"
config.TEST_DIR = config.DATA_DIR / "test"

CKPT_RESNET50 = DOWNLOADABLES / "resnet50_weights.pth"
CKPT_EFFICIENTNET_B0 = DOWNLOADABLES / "efficientnet_b0_weights.pth"
CKPT_EFFICIENTNET_V2_S = DOWNLOADABLES / "efficientnet-v2-s_weights.pth"

config.CKPT_PATH = CKPT_EFFICIENTNET_V2_S

print("DATA_DIR :", config.DATA_DIR)
print("Device   :", config.DEVICE)

100%|██████████| 3.35G/3.35G [00:44<00:00, 81.1MB/s]

Extracting files...


split downloaded to: /root/.cache/kagglehub/datasets/delno002/split-ffpp/versions/1
DATA_DIR : /root/.cache/kagglehub/datasets/delno002/split-ffpp/versions/1
Device   : cpu


## 3. Imports

Everything below comes from the project's `src/` package

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.dataset import FFPPFrameDataset, get_test_transform, count_split
from src.model import load_model, build_classifier, EfficientNetV2SFeatureExtractor
from src.inference import (
    evaluate_frame_classifier,
    run_frame_inference,
    aggregate_to_videos,
)
from src.features import extract_video_features
from src.classifiers import build_classifiers, fit_scaler, train_and_evaluate
from src.evaluation import print_metrics, plot_confusion, plot_confusions_grid

# gpu if available, otherwise cpu
DEVICE = config.DEVICE
print("Imported. Device =", DEVICE)

Imported. Device = cpu


## 5. Phase 1, Architecture Comparison

We fine-tuned three ImageNet-pretrained backbones on the FF++ training split with the same training schedule:

- Frozen backbone for the first 3 epochs (head-only training).
- Unfreeze the deeper blocks at epoch 3 and continue at a lower learning rate.
- Class-weighted CrossEntropyLoss to compensate for the imbalance.
- Early stopping on validation accuracy with patience = 5.

The training itself happened on Kaggle (`notebooks/architecture_training/`). This notebook loads the saved checkpoints and evaluates them on the test split.

### 5.1 What we trained

ResNet-50, EfficientNet-B0, EfficientNet-V2-S were all pretrained on the ImageNet dataset and the the trainable head was replaced by BatchNorm1d, Dropout(0.3) and Linear(2048, 2) for all three backbones.

### 5.2 Evaluating each checkpoint on the test set

We load each checkpoint in turn, run it over every frame in the test split, and report accuracy, per-class precision/recall/F1, and the confusion matrix.

In [ ]:
# build the test dataset once - all three architectures see the same frames
# in the same order, so the comparison is apples to apples
test_dataset = FFPPFrameDataset(config.TEST_DIR, transform=get_test_transform())
print("Total test frames:", len(test_dataset))

In [ ]:
# list the three architectures with their checkpoint paths
arch_specs = [
    ("resnet50", CKPT_RESNET50),
    ("efficientnet_b0", CKPT_EFFICIENTNET_B0),
    ("efficientnet_v2_s", CKPT_EFFICIENTNET_V2_S),
]

arch_results = {}

for arch, ckpt_path in arch_specs:
    print("Checkpoint:", ckpt_path)

    # build the architecture and load the saved weights
    model = load_model(arch=arch, ckpt_path=ckpt_path, device=DEVICE)

    # run the model over every test frame, get true labels and predicted labels
    y_true, y_pred = evaluate_frame_classifier(
        model, test_dataset,
        device=DEVICE,
        batch_size=config.BATCH_SIZE,
        num_workers=config.NUM_WORKERS,
    )

    # save for later plotting
    arch_results[arch] = {"y_true": y_true, "y_pred": y_pred}

    print_metrics(y_true, y_pred, title=f"Test metrics - {arch}")

    # free gpu memory before loading the next checkpoint
    del model
    import torch, gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

### 5.3 Confusion matrices side by side

In [ ]:
# plot all three confusion matrices in a single row for easy comparison
plot_confusions_grid([
    ("ResNet-50", arch_results["resnet50"]["y_true"], arch_results["resnet50"]["y_pred"]),
    ("EfficientNet-B0", arch_results["efficientnet_b0"]["y_true"], arch_results["efficientnet_b0"]["y_pred"]),
    ("EfficientNet-V2-S", arch_results["efficientnet_v2_s"]["y_true"], arch_results["efficientnet_v2_s"]["y_pred"]),
])

### 5.4 Picking a winner

EfficientNet-V2-S achieves the highest test accuracy of the three architectures (this matches the recorded results in the original training notebooks: ~92% test accuracy for EfficientNet-V2-S vs ~85% for EfficientNet-B0 and ~90% for ResNet-50). It is also the most modern of the three, with training-aware NAS designed for both accuracy and training speed.

We use **EfficientNet-V2-S** for the two video-level experiments below.

## 6. Phase 2: Frame-Level Inference and Video-Level Aggregation

The architecture comparison above operates at the frame level: every frame is classified in isolation and we report per-frame accuracy. But our real task is *video-level*. We want one decision per video, not one per frame.

We compare two video-level aggregation strategies, both built on top of the fine-tuned EfficientNet-V2-S classifier:

- **Majority voting (hard).** Take the per-frame `argmax` and vote.
- **Average softmax (soft).** Average the per-frame `softmax` probabilities and `argmax` the result.

Hard voting is robust to a few wildly confident frames flipping the decision but throws away confidence information. Soft averaging keeps the confidence signal at the cost of being more sensitive to a small number of high-confidence wrong frames.

### 6.1 Load the EfficientNet-V2-S classifier

In [ ]:
# load the fine-tuned efficientnet-v2-s with its 2-class head
classifier = build_classifier(
    ckpt_path=CKPT_EFFICIENTNET_V2_S,
    device=DEVICE,
    num_classes=2,
)

# quick sanity check on model size
n_params = sum(p.numel() for p in classifier.parameters())
print(f"Loaded EfficientNet-V2-S - {n_params/1e6:.2f}M parameters")

### 6.2 Frame-level inference

Run the classifier over every frame in the test split. The resulting DataFrame has one row per frame and includes the per-class softmax probabilities, which we'll need for soft aggregation.

In [ ]:
# run inference on every test frame
# the returned dataframe has one row per frame with both the predicted label
# and the raw softmax probabilities (we need probs for soft aggregation later)
frame_df = run_frame_inference(
    classifier,
    test_dataset,
    device=DEVICE,
    batch_size=config.BATCH_SIZE,
    num_workers=config.NUM_WORKERS,
)

print("Frame-level DataFrame:", frame_df.shape)
frame_df.head()

In [ ]:
# save the per-frame predictions to disk for later inspection
frame_csv_path = config.OUTPUT_DIR / "efficientnet_v2_s_frame_scores.csv"
frame_df.to_csv(frame_csv_path, index=False)
print("Saved:", frame_csv_path)

### 6.3 Aggregate to video-level

Group frames by `video_id` and compute both aggregations.

In [ ]:
# group frames by video and apply both aggregation methods
# the dataframe has one row per video with majority-vote and avg-softmax predictions
video_df = aggregate_to_videos(frame_df)

print("Number of test videos:", len(video_df))
video_df.head()

In [ ]:
# save the per-video predictions for inspection
video_csv_path = config.OUTPUT_DIR / "efficientnet_v2_s_video_predictions.csv"
video_df.to_csv(video_csv_path, index=False)
print("Saved:", video_csv_path)

### 6.4 Frame-count distribution per video

How many frames does a typical test video contribute? Aggregation is only meaningful if videos contribute enough frames for a vote / average to be more than a coin flip.

In [ ]:
# how many frames does each video contribute? (mean / min / max / quartiles)
frame_counts = frame_df.groupby("video_id").size()
print(frame_counts.describe())

# how many unique videos are there per class?
print("\nUnique videos per class:")
print(frame_df.groupby("true_label")["video_id"].nunique())

### 6.5 Video-level metrics by Majority Voting

In [ ]:
print_metrics(
    video_df["true_label_idx"],
    video_df["majority_pred_idx"],
    title="Video-level - Majority Voting",
)

# confusion matrix for the same
plot_confusion(
    video_df["true_label_idx"],
    video_df["majority_pred_idx"],
    title="Confusion Matrix - Majority Voting",
)
plt.show()

### 6.6 Video-level metrics for Average Softmax

In [ ]:
# same thing but using the average-softmax aggregation
print_metrics(
    video_df["true_label_idx"],
    video_df["avg_pred_idx"],
    title="Video-level - Average Softmax",
)

plot_confusion(
    video_df["true_label_idx"],
    video_df["avg_pred_idx"],
    title="Confusion Matrix - Average Softmax",
)
plt.show()

### 6.7 How often do the two aggregations disagree?

In [ ]:
# how often do the two aggregations give the same answer?
same = (video_df["majority_pred_idx"] == video_df["avg_pred_idx"]).mean()
n_diff = int((video_df["majority_pred_idx"] != video_df["avg_pred_idx"]).sum())

print(f"Fraction of videos where the two aggregations agree: {same:.4f}")
print(f"Number of videos where they disagree: {n_diff} / {len(video_df)}")

## 7. Phase 3: Backbone as Feature Extractor + Classical ML

Instead of relying on the fine-tuned classification head, we now treat EfficientNet-V2-S as a frozen 1280-D feature extractor (everything up to and including global average pooling) and train classical classifiers on top.

Per-video pipeline:

1. Run every frame through the frozen backbone. 1280-D embedding.
2. Average those frame embeddings, one 1280-D vector per video.
3. Standardize features (`StandardScaler`).
4. Fit SVM / MLP / Logistic Regression on the *training* video embeddings.
5. Evaluate on the *test* video embeddings.

### 7.1 Build the feature extractor

Same checkpoint as above but only the classification head is dropped.

In [ ]:
# same checkpoint as in section 6, but the classification head is dropped
feature_extractor = EfficientNetV2SFeatureExtractor(
    ckpt_path=CKPT_EFFICIENTNET_V2_S,
    num_classes=2,
    device=DEVICE,
).to(DEVICE)

feature_extractor.eval()
print("Feature extractor ready.")

### 7.2 Extract video-level features for `train` and `test`


In [ ]:
# build train and test datasets (same transform, no augmentation)
train_dataset = FFPPFrameDataset(config.TRAIN_DIR, transform=get_test_transform())
test_dataset_for_feats = FFPPFrameDataset(config.TEST_DIR, transform=get_test_transform())

# embed every train frame, then average per video to get one vector per video
X_train, y_train, train_video_ids = extract_video_features(
    train_dataset,
    feature_extractor,
    device=DEVICE,
    batch_size=config.BATCH_SIZE,
    num_workers=config.NUM_WORKERS,
)
print("Train  features:", X_train.shape, "labels:", y_train.shape)

# same for test
X_test, y_test, test_video_ids = extract_video_features(
    test_dataset_for_feats,
    feature_extractor,
    device=DEVICE,
    batch_size=config.BATCH_SIZE,
    num_workers=config.NUM_WORKERS,
)
print("Test   features:", X_test.shape, "labels:", y_test.shape)

### 7.3 Standardize features

In [ ]:
# fit a standardscaler on the training features and apply it to both splits
scaler = fit_scaler(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("Scaled shapes:", X_train_scaled.shape, X_test_scaled.shape)

### 7.4 Train and evaluate SVM / MLP / LogReg

In [ ]:
# fit svm, mlp, logreg on the scaled training features and evaluate on the test features
clf_results = train_and_evaluate(
    build_classifiers(random_state=42),
    X_train_scaled, y_train,
    X_test_scaled,  y_test,
)

# print accuracy and full classification report for each one
for name, res in clf_results.items():
    print(f"Accuracy: {res['accuracy']:.4f}")
    print(res["report"])

### 7.5 Confusion matrices side by side

In [ ]:
plot_confusions_grid([
    ("SVM", y_test, clf_results["SVM"]["y_pred"]),
    ("LogReg", y_test, clf_results["LogReg"]["y_pred"]),
    ("MLP", y_test, clf_results["MLP"]["y_pred"]),
])

### 7.6 Persist trained classifiers

In [ ]:
# save the fitted classifiers + scaler so we can reload them later without retraining
import joblib

# put everything in one folder
save_dir = config.OUTPUT_DIR / "saved_pipeline"
save_dir.mkdir(parents=True, exist_ok=True)

# what to save
artifacts = {
    "svm_model.pkl": clf_results["SVM"]["model"],
    "mlp_model.pkl": clf_results["MLP"]["model"],
    "logreg_model.pkl": clf_results["LogReg"]["model"],
    "scaler.pkl": scaler,
    "class_to_idx.pkl": config.CLASS_TO_IDX,
    "idx_to_class.pkl": config.IDX_TO_CLASS,
}

# dump each one to disk
for filename, obj in artifacts.items():
    joblib.dump(obj, save_dir / filename)

# list what got saved
print("Saved files in", save_dir, ":")
for f in sorted(save_dir.iterdir()):
    print(" -", f.name)

## 8. Interpretation

**Architecture comparison.** EfficientNet-V2-S is the strongest of the three backbones on this test set, with both the highest accuracy and the most balanced per-class precision/recall. ResNet-50 is a close second; EfficientNet-B0 is meaningfully behind. This ordering is consistent with the published numbers on ImageNet and aligns with the parameter-count / capacity ordering of the three models. It is also visible in validation accuracy curves recorded in the training notebooks: EfficientNet-V2-S converges to ~94% validation accuracy by epoch 25, while EfficientNet-B0 plateaus around 88% and ResNet-50 around 91%.

**Frame-level vs video-level.** Once we move from per-frame to per-video evaluation, accuracy goes up - a single confidently misclassified frame no longer counts as a mistake at the video level if the rest of the video votes the other way. This is a property of the data more than the method: most FF++ videos are uniformly real or uniformly manipulated, so collapsing many independent frame predictions into one decision is a strong noise-reduction operation.

**Soft vs hard aggregation.** Average-softmax aggregation is consistently at least as accurate as majority voting on this split. The reason is that voting throws away confidence - a frame that the model is 0.51 confident about and one it is 0.99 confident about count the same. Soft averaging keeps the confidence signal, which helps on the small subset of videos where per-frame predictions are genuinely mixed. The two strategies agree on the vast majority of videos (see Section 6.7); they only diverge on the borderline cases.

**Classical classifiers on top of the frozen backbone.** SVM, MLP, and Logistic Regression trained on mean-pooled embeddings perform competitively with the fine-tuned head. This indicates that the heavy lifting is being done by the *representation* (the fine-tuned backbone), not the final classifier - once you have a good 1280-D embedding, the final binary decision is approximately linearly separable.

## 9. Limitations and Failures

We did not get every idea to work, and we want to be explicit about what those were.

**Same-dataset evaluation.** Train and test both come from FF++. We did not run a cross-dataset evaluation (e.g. train on FF++, test on Celeb-DF or DFDC) within the project timeline. A high test number on FF++ is therefore not a strong claim about generalization to deepfakes produced by other generators or pipelines.

**Mean pooling discards temporal structure.** Averaging frame embeddings ignores their order. That is fine for FF++, where every frame of a manipulated video is independently manipulated, but it would be a poor choice for forgeries whose tell-tale signature is *temporal inconsistency* (flicker between adjacent frames, identity drift). A recurrent or attention-based aggregator would be needed there.

**Frame sampling, not whole-video processing.** We process precomputed face crops, not raw videos. Detection / alignment errors upstream of this notebook can propagate silently. Several "wrong" videos we spot-checked had crops where the face was poorly centered.

**Class imbalance partially addressed.** During training we used a class-weighted CrossEntropyLoss. We did not, however, apply oversampling or threshold tuning at inference time. The minority (real) class therefore still sees lower recall in some methods.

**Hyperparameter search was minimal.** We used reasonable defaults for SVM-RBF, MLP, and LogReg rather than running a grid search. The relative ordering of these classifiers might shift slightly with proper tuning.

**Did not retrain inside this notebook.** Each fine-tuning run takes several hours on a T4. We instead saved checkpoints from Kaggle runs (preserved in `notebooks/architecture_training/`) and load them here. The grader can verify the test-set numbers but cannot rerun training without a multi-hour GPU session.

## 10. Conclusions

- Among the three pretrained backbones we tested, **EfficientNet-V2-S** is the strongest fine-tuned frame classifier on FF++.
- Aggregating per-frame predictions to per-video decisions improves accuracy. **Soft averaging** of softmax probabilities is mildly but consistently better than hard majority voting, and is the most cost-effective option among the ones we tested.
- Replacing the fine-tuned head with classical classifiers (SVM / MLP / LogReg) on mean-pooled embeddings gives **competitive** results, confirming that the hard work is done by the representation, not the final classifier.
- The largest practical gains, if we were continuing this project, would come from cross-dataset evaluation, a temporally-aware aggregator that does not collapse the frame dimension by simple averaging, and explicit handling of class imbalance at inference time.